# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's performance on one day, for one client
(the grain of fact_content_daily_performance: report_date × client ×
content). I queried the full table filtered to a single mid-panel
month, month=2026-03, to avoid the trap of developing label logic on
the final month (which is the natural outcome window for any
past→future label).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label (what I predict): is_declining_label, built from comparing
gsc_impressions in a last-30-day window vs. the prior-30-day window
(declining if last30 < 80% of prev30). This table has no pre-made
trend_direction column, so the label is constructed directly from
raw impressions — an observed outcome, not a rule someone else
already applied.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!pip install -q huggingface_hub pandas pyarrow

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

print("Logged in to Hugging Face.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.


In [2]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# --- Query 1: Grain check ---
# one row should be one (report_date, client_hash_id, content_hash_id)
dupe_check = con.sql(f"""
    SELECT COUNT(*) AS n_dupes FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {fact_daily}
        WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
    )
""").df()
print("Query 1 — Grain check (should be 0 duplicate combos)")
print(dupe_check)
print()

# --- Query 2: Row count + date span for March 2026 ---
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {fact_daily}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print("Query 2 — Row count and date span (March 2026)")
print(counts)
print()

# --- Query 3: Availability filter ---
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {fact_daily}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()
print("Query 3 — Availability (GA4)")
print(avail)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain check (should be 0 duplicate combos)
   n_dupes
0        0

Query 2 — Row count and date span (March 2026)
    n_rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 — Availability (GA4)
   total_rows  ga4_available_rows
0     9841378            413966.0


Five features, max, for this lane:

1. gsc_impressions (prev 30d) — available when: end of each prior
   30-day window, before the decision moment.
2. gsc_avg_position (prev 30d) — available when: same prior window.
3. days_since_last_update — available when: known at any point,
   depends only on content history.
4. ga4_data_available (flag) — available when: known at ingestion,
   decides whether engagement features are usable for a row.
5. client tenure (days since gsc_data_start) — available when: known
   from dim_clients at any point.

In [3]:
schema = con.sql(f"DESCRIBE SELECT * FROM {fact_daily}").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [4]:
import pandas as pd

# Pull raw daily data for a wider window so we can compare last-30 vs prev-30
raw = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date,
           gsc_impressions, gsc_clicks, gsc_avg_position
    FROM {fact_daily}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-04-01'
""").df()

raw["report_date"] = pd.to_datetime(raw["report_date"])
end_d = raw["report_date"].max()

raw["is_last30"] = raw["report_date"] > (end_d - pd.Timedelta(days=30))
raw["is_prev30"] = ~raw["is_last30"]

agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: pd.Series({
        "imp_last30": g.loc[g["is_last30"], "gsc_impressions"].sum(),
        "imp_prev30": g.loc[g["is_prev30"], "gsc_impressions"].sum(),
        "clk_last30": g.loc[g["is_last30"], "gsc_clicks"].sum(),
        "pos_prev30": g.loc[g["is_prev30"], "gsc_avg_position"].mean(),
    })
).reset_index()

agg = agg[agg["imp_prev30"] >= 100]  # enough history to define a meaningful trend
agg["is_declining_label"] = (agg["imp_last30"] < 0.8 * agg["imp_prev30"]).astype(int)

print(f"{len(agg):,} content items with enough history")
agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

81,383 content items with enough history


/tmp/ipykernel_5401/1523949790.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_prev30,is_declining_label
7,client_0797ff3a1fc9a6a5,content_04c67f3541177192,325.0,252.0,2.0,18.820788,0
8,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,26.0,144.0,0.0,9.717545,1
14,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,141.0,125.0,0.0,8.986303,0
18,client_0797ff3a1fc9a6a5,content_1207efddce873942,446.0,189.0,0.0,13.785365,0
22,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,229.0,167.0,0.0,11.424053,0


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# HONEST features: only prior-window signals, before the outcome window
honest_features = ["imp_prev30", "pos_prev30"]

# LEAKY features: sneak in clk_last30, which comes from the SAME window
# the label (is_declining_label) is defined on — clicks and impressions
# in that window are almost mechanically linked
leaky_features = honest_features + ["clk_last30"]

def quick_score(features):
    d = agg.dropna(subset=features + ["is_declining_label"])
    X, y = d[features], d["is_declining_label"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_score(honest_features)
leaky_auc = quick_score(leaky_features)

print("Honest AUC (no leakage):", honest_auc)
print("Leaky AUC (clk_last30 included):", leaky_auc)

Honest AUC (no leakage): 0.5400715831930339
Leaky AUC (clk_last30 included): 0.6734896504147823


Adding clk_last30 as a feature lifts the AUC from 0.540 to 0.673 — a
real, suspicious jump. The reason: clk_last30 comes from the exact same
30-day window the label (is_declining_label) is computed on, so clicks
and impressions in that window are mechanically related to the label
by construction, not because the model learned a generalizable pattern.
I remove clk_last30 and keep only imp_prev30 and pos_prev30 — signals
that are fully known before the outcome window starts. The honest AUC
of 0.540 is the real, defensible number; it's weak, but it's not a lie.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has real limits. Only ~4.2% of March 2026 rows have GA4 data
available (413,966 of 9,841,378) — most rows are GSC-only, so engagement
signals like scroll_rate can't be used for most of the panel without
dropping the majority of data. History depth also differs per client
(an unbalanced panel), so a fixed calendar window like March 2026 doesn't
mean equal history for every client — some may have joined GSC/GA4 tracking
partway through. Finally, this is a single month; seasonal or one-off
events in March wouldn't be visible, and any window-based label (e.g.
30-day trend) risks overlapping with adjacent months at the edges.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.